# Do 3D methods resolve vertical doublets?
ovrlpy marks vertical signal mixing as low VSI (vertical signal integrity below 0.7). If 3D methods separated stacked cells,
their footprints should overlap where VSI is low.

1. Cell overlap ranges from 0.1% of segmented area (2D control) to 35% (Baysor 3D).
2. Overlaps are enriched for low VSI only in Baysor 3D (1.6-fold) and vpt 3D P and P+N. For Proseg and SIS the direction depends on the cohort.
3. Overlaps are thin slivers at cell borders. At most 0.2% of low-VSI signal lies where a top cell sits on a bottom cell.
4. Most cells span 4 to 7 of the 7 z planes, so the 10 µm section is about one cell thick.

VSI therefore needs no 3D correction.

28 samples from aging, ABCAtlas and VizgenMouseBrain. Cells pass the merge QC (spatial outliers, counts, genes, volume) unless labelled pre-QC.

In [ ]:
cohorts = ["aging", "ABCAtlas", "VizgenMouseBrain"]
controls = ["Cellpose_1_DAPI_PolyT"]
show_sample, crop = "ABCAtlas_s4_r0", 300
low_vsi, min_signal = 0.7, 3           # ovrlpy detect_doublets defaults
offset = 13                            # as in metrics/ovrl.py

import pathlib, warnings, yaml
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.font_manager
from matplotlib.colors import ListedColormap
import matplotlib.path as mpath
from rasterio.features import rasterize
from rasterio.enums import MergeAlg
from scipy.ndimage import label, uniform_filter
from cellseg_benchmark import BASE_PATH, _constants
from cellseg_benchmark.adata_utils import filter_low_quality_cells
import geopandas as gpd, inspect, zarr
from anndata.io import read_elem

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = "retina"
mm = 1 / 25.4
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 7, "axes.titlesize": 7, "axes.labelsize": 7, "xtick.labelsize": 6, "ytick.labelsize": 6,
    "legend.fontsize": 6, "legend.frameon": False, "axes.linewidth": 0.5,
    "xtick.major.width": 0.5, "ytick.major.width": 0.5, "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 200, "pdf.fonttype": 42,
})
base = pathlib.Path(BASE_PATH)
for f in (base / "misc").glob("*.ttf"):
    matplotlib.font_manager.fontManager.addfont(str(f))
npz_file = "vertical_doublets_ovrlpy_output.npz"
meta = yaml.safe_load(open(base / "misc" / "sample_metadata.yaml"))
samples = sorted(s for s, v in meta.items()
                 if v.get("cohort") in cohorts and s not in yaml.safe_load(open(base / "misc" / "samples_excluded.yaml")).get(v.get("cohort"), [])
                 and (base / "samples" / s / npz_file).exists())
bins = np.linspace(0, 1, 51)
qc_thr = {k: v.default for k, v in inspect.signature(filter_low_quality_cells).parameters.items()
      if k in ("min_counts", "min_genes", "min_volume_threshold", "max_volume_fold")}


def methods_for(sample):
    shapes = base / "samples" / sample / "sdata_z3.zarr" / "shapes"
    have = sorted(p.name.removeprefix("boundaries_") for p in shapes.glob("boundaries_*")) if shapes.is_dir() else []
    return [m for m in have if m in controls or m.startswith(tuple(_constants.methods_3D))]


print(len(samples), "samples")

In [ ]:
zcols = ("ZIndex", "z_plane", "layer", "ZLevel", "z")


def cell_table(sample, method, ids):
    """Merge QC (spatial outliers, then counts/genes/volume) and cell type per cell, indexed by the table ids that best match `ids`."""
    g = zarr.open_group(str(base / "samples" / sample / "sdata_z3.zarr" / "tables" / f"adata_{method}"), mode="r")
    counts, obs = read_elem(g["X"]), read_elem(g["obs"])
    key = max([obs.index.astype(str)] + ([obs["cell_id"].astype(str)] if "cell_id" in obs else []),
              key=lambda k: pd.Index(ids).isin(k).mean())
    outlier = np.zeros(len(obs), bool)
    lasso = base / "samples" / sample / "cell_outlier_coordinates.csv"
    if lasso.exists():
        sel = pd.read_csv(lasso, skiprows=4)
        t = np.loadtxt(pathlib.Path(meta[sample]["path"], "images", "micron_to_mosaic_pixel_transform.csv")).reshape(3, 3)
        sel[["X", "Y"]] -= t[[0, 1], 2] / t[[0, 1], [0, 1]]
        xy = np.asarray(read_elem(g["obsm"]["spatial_microns" if "spatial_microns" in g["obsm"] else "spatial"]))[:, :2]
        for _, poly in sel.groupby("Selection"):
            if len(poly) > 2:
                outlier |= mpath.Path(poly[["X", "Y"]].to_numpy()).contains_points(xy)
    low = (np.ravel(counts.sum(axis=1)) < qc_thr["min_counts"]) | (np.ravel((counts > 0).sum(axis=1)) < qc_thr["min_genes"])
    vol = obs["volume_final"].to_numpy()
    ok = ~outlier & ~low & (vol >= qc_thr["min_volume_threshold"]) & (vol <= qc_thr["max_volume_fold"] * np.median(vol[~outlier & ~low]))
    return pd.DataFrame({"keep": ok, "cell_type": obs["cell_type_revised"].astype(str).to_numpy()}, index=np.asarray(key))


def load_cells(sample, method, tf, planes=False, bbox=None):
    """From the master sdata: footprint per cell in ovrlpy pixels, z per polygon (None if flattened),
    and per cell merge-QC pass + cell type. planes=True: per-plane polygons (cell, z, cell_type, keep) only. bbox=(x0, y0, x1, y1) in ovrlpy px: only polygons in it."""
    master = base / "samples" / sample / "sdata_z3.zarr"
    gdf = gpd.read_parquet(next((master / "shapes" / f"boundaries_{method}").glob("*.parquet")))
    to_micron = np.linalg.inv(tf) if method.startswith(tuple(_constants.image_based)) and method != "Cellpose_1_Merlin" else np.eye(3)
    aff = np.array([[1, 0, tf[0, 2] / tf[0, 0] - offset], [0, 1, tf[1, 2] / tf[1, 1] - offset], [0, 0, 1]]) @ to_micron
    gdf = gdf.set_geometry(gdf.geometry.affine_transform([*aff[:2, :2].ravel(), *aff[:2, 2]]))
    if bbox is not None:
        gdf = gdf.cx[bbox[0]:bbox[2], bbox[1]:bbox[3]]
    ids = (gdf["cell_id"] if "cell_id" in gdf else gdf.index.to_series()).astype(str).to_numpy()
    zc = next((c for c in zcols if c in gdf.columns), None)

    cells = cell_table(sample, method, ids)

    if planes:
        if zc is None:
            return None
        return gpd.GeoDataFrame({"cell": ids, "z": pd.to_numeric(gdf[zc], errors="coerce").to_numpy(),
                                 "cell_type": cells.cell_type.reindex(ids).to_numpy(),
                                 "keep": cells.keep.reindex(ids).fillna(False).to_numpy(bool)},
                                geometry=gdf.geometry.to_numpy())

    g = gpd.GeoDataFrame({"cid": ids}, geometry=gdf.geometry.to_numpy())
    dup = g.cid.duplicated(keep=False)
    foot = pd.concat([g[~dup].set_index("cid").geometry, g[dup].dissolve(by="cid").geometry])
    cells = cells.reindex(foot.index).fillna({"keep": False})
    z = None if zc is None else pd.DataFrame({"cell": ids, "z": pd.to_numeric(gdf[zc], errors="coerce").to_numpy()})
    return foot, z, cells


def count_map(geoms, shape):
    geoms = [(g, 1) for g in geoms if g is not None and not g.is_empty]
    if not geoms:
        return np.zeros(shape, np.int16)
    return rasterize(geoms, out_shape=shape, merge_alg=MergeAlg.add, dtype=np.int16)


def overlap_stats(foot, z, vsi, signal):
    k = count_map(foot.values, vsi.shape)
    seg, multi = k >= 1, k >= 2
    valid = signal >= min_signal
    low = valid & (vsi < low_vsi)
    out = {
        "n_cells": len(foot),
        "k2_area": multi[seg].mean(),
        "k3_area": (k >= 3)[seg].mean(),
        "k2_signal": signal[multi].sum() / signal[seg].sum(),
        "vsi_k1": np.median(vsi[seg & ~multi & valid]),
        "vsi_k2": np.median(vsi[multi & valid]) if (multi & valid).any() else np.nan,
        "patch_px": np.median(np.bincount(label(multi)[0].ravel())[1:]) if multi.any() else np.nan,
        "lowvsi_cov": signal[seg & low].sum() / signal[low].sum(),
        "vsi_hist": np.stack([np.histogram(vsi[valid & (k.clip(0, 3) == j)], bins)[0] for j in range(4)]),
    }
    if z is None:
        return out

    poly_z = z["z"].value_counts()
    z = z.groupby("cell")["z"].agg(["min", "max", "nunique"]).reindex(foot.index).dropna()
    mid = (z["min"].min() + z["max"].max()) / 2
    top, bot = z.index[z["min"] > mid], z.index[z["max"] < mid]
    zs = (count_map(foot.loc[top].values, vsi.shape) >= 1) & (count_map(foot.loc[bot].values, vsi.shape) >= 1)
    out.update(
        z_matched=len(z) / len(foot),
        top_only=len(top) / len(z),
        bottom_only=len(bot) / len(z),
        zsplit_signal=signal[zs].sum() / signal[seg].sum(),
        lowvsi_recall=signal[zs & low].sum() / signal[low].sum(),
        n_planes=z["nunique"].value_counts(),
        poly_z=poly_z,
    )
    return out

In [ ]:
%%time
cache = base / "misc" / "tmp" / "z_overlap_k_master.pkl"
res = pd.read_pickle(cache) if cache.exists() else {}

for s in samples:
    todo = [m for m in methods_for(s) if (s, m, "pre") not in res]
    if not todo:
        continue
    npz = np.load(base / "samples" / s / npz_file)
    vsi, signal = npz["integrity"].astype(np.float32), npz["signal"].astype(np.float32)
    tf = pd.read_csv(pathlib.Path(meta[s]["path"], "images/micron_to_mosaic_pixel_transform.csv"),
                     sep=" ", header=None).values
    print(s, flush=True)
    for m in todo:
        try:
            foot, z, cells = load_cells(s, m, tf)
            keep = cells.keep.to_numpy(bool)
            print(f"  {m}: {keep.mean():.0%} pass QC, {cells.cell_type.notna().mean():.0%} with cell type")
            for qc, f in [("pre", foot), ("post", foot[keep])]:
                res[(s, m, qc)] = overlap_stats(f, None if z is None else z[z.cell.isin(f.index)], vsi, signal)
        except Exception as e:
            print(f"  {m} ! {type(e).__name__}: {e}")
    pd.to_pickle(res, cache)

df = pd.DataFrame.from_dict(res, orient="index").rename_axis(["sample", "method", "qc"]).reset_index()
df = df[df["sample"].isin(samples) & (df.method.str.startswith(tuple(_constants.methods_3D)) | df.method.isin(controls))]
pairs = df.loc[df.qc == "post", ["sample", "method"]]
print(f"{len(pairs)} sample-method pairs; pre-QC uses the same pairs")
df = df.merge(pairs, on=["sample", "method"])
df["dvsi"] = df.vsi_k2 - df.vsi_k1

method_label = {m: "2D Ctrl / CP1 P" if m in controls else _constants.method_names.get(m, m) for m in df.method.unique()}
method_color = {m: _constants.method_colors.get(m, "0.4") for m in df.method.unique()}
df["group"] = df.method.map(method_label)
groups = df[df.qc == "post"].groupby("group").k2_signal.median().sort_values(ascending=False).index

per_method = df.drop(columns=["vsi_hist", "n_planes", "poly_z"], errors="ignore").groupby(["method", "qc"]).median(numeric_only=True)
order = per_method.xs("post", level="qc").sort_values("k2_signal", ascending=False).index
per_method.loc[order, ["n_cells", "k2_signal", "dvsi", "patch_px", "zsplit_signal", "lowvsi_recall"]].round(4)

## 1. Z integrity and cell overlap
1 µm bins with at least 3 transcripts, pooled over samples.
**a,b** Cells per bin in high- and low-VSI area, bins without cells excluded. **c-e** VSI in bins with 0 (control), 1 and 2 or more cells.
**f** Area per sample with 1 and with 2 or more cells, labelled with the overlap share.

In [ ]:
post = df[df.qc == "post"]
hist = {g: np.sum(post.loc[post.group == g, "vsi_hist"].tolist(), axis=0) for g in groups}
n_runs = post.group.value_counts()
lowbin = bins[:-1] < low_vsi
vsi_bin = np.digitize(bins[:-1], [0.5, low_vsi, 0.9])

a_hi, a_lo = (pd.DataFrame({g: h[1:, sel].sum(axis=1) for g, h in hist.items()}, index=["1", "2", "≥3"]).T
              for sel in (~lowbin, lowbin))
vsi_labels = ["<0.5", f"0.5–{low_vsi}", f"{low_vsi}–0.9", "≥0.9"]
b0, b1, b2 = (pd.DataFrame({g: np.bincount(vsi_bin, weights=sel(h), minlength=4) for g, h in hist.items()}, index=vsi_labels).T
              for sel in (lambda h: h[0], lambda h: h[1], lambda h: h[2:].sum(axis=0)))
a_hi, a_lo, b0, b1, b2 = (100 * d.div(d.sum(axis=1), axis=0) for d in (a_hi, a_lo, b0, b1, b2))
px = pd.DataFrame({g: [h[1].sum(), h[2:].sum()] for g, h in hist.items()}, index=["k1", "k2"]).T.div(n_runs, axis=0)
share = 100 * px.k2 / (px.k1 + px.k2)
groups = share.sort_values(ascending=False).index

k_cols = ["#9e9e9e", "#f28e2b", "#c0392b"]
vsi_cols = ["#b2182b", "#ef8a62", "#92c5de", "#2166ac"]
panels = [
    (a_hi, k_cols, "Cells per 1 µm bin", f"% of area with high z integrity\n(VSI ≥ {low_vsi})"),
    (a_lo, k_cols, None, f"% of area with low z integrity\n(VSI < {low_vsi})"),
    (b0, vsi_cols, "Vertical signal integrity (VSI)", "% of area with 0 cells"),
    (b1, vsi_cols, None, "% of area with 1 cell"),
    (b2, vsi_cols, None, "% of area with ≥ 2 cells"),
]
y = np.arange(len(groups))
fig, axs = plt.subplots(1, 6, figsize=(230 * mm, (18 + 5 * len(groups)) * mm), sharey=True,
                        gridspec_kw={"width_ratios": [1, 1, 1, 1, 1, 0.9]})
for ax, (d, cols, title, xlabel) in zip(axs, panels):
    left = np.zeros(len(d))
    for col, colour in zip(d.columns, cols):
        ax.barh(y, d.loc[groups, col], left=left, height=0.72, color=colour, lw=0, label=col)
        left += d.loc[groups, col].to_numpy()
    ax.set(xlim=(0, 100), xlabel=xlabel)
    if title:
        ax.legend(title=title, ncol=3 if len(cols) == 3 else 2, loc="lower left", bbox_to_anchor=(0, 1.01), handlelength=1,
                  handletextpad=0.4, columnspacing=0.8, borderaxespad=0, alignment="left")

ax = axs[5]
ax.hlines(y, px.loc[groups, "k2"], px.loc[groups, "k1"], color="0.8", lw=1.2, zorder=1)
ax.scatter(px.loc[groups, "k1"], y, s=14, color="#9e9e9e", lw=0, zorder=2, label="1")
ax.scatter(px.loc[groups, "k2"], y, s=14, color="#f28e2b", lw=0, zorder=2, label="≥ 2")
for yi, g in zip(y, groups):
    ax.text(px.loc[g, "k2"] / 1.6, yi, f"{share[g]:.1f}%", va="center", ha="right", fontsize=6)
ax.set(xscale="log", xlabel="Area per sample (µm²)")
ax.set_xlim(left=ax.get_xlim()[0] / 8)
ax.legend(title="Cells per 1 µm bin", ncol=2, loc="lower left", bbox_to_anchor=(0, 1.01), handlelength=1,
          handletextpad=0.3, columnspacing=0.8, borderaxespad=0, alignment="left")

for ax in axs:
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)
axs[0].set_yticks(y, groups)
axs[0].invert_yaxis()
fig.tight_layout()
fig.subplots_adjust(wspace=0.12)
plt.show()
pd.concat({"high VSI by k": a_hi, "low VSI by k": a_lo, "k=0 by VSI": b0, "k=1 by VSI": b1, "k≥2 by VSI": b2}, axis=1).round(1)

### Overlap and low-VSI enrichment per sample
**a** Share of segmented area covered by two or more cells. **b** Log2 fold change of the low-VSI share in overlap area versus
single-cell area; 0 means no enrichment. Samples with less than 1000 µm² overlap are left out. Dots are samples, bars medians.
Wilcoxon signed-rank test against 0, Benjamini-Hochberg corrected across methods (* q < 0.05, ** q < 0.01, *** q < 0.001).

The table also lists bootstrap 95% CIs of the median and paired tests against the 2D control and between overlap and single-cell area.

In [ ]:
import matplotlib.ticker
from scipy.stats import wilcoxon, false_discovery_control

lowbin = bins[:-1] < low_vsi
rows = []
for r in df[df.qc == "post"].itertuples():
    n, low = r.vsi_hist.sum(axis=1), r.vsi_hist[:, lowbin].sum(axis=1)
    rows.append({"sample": r.sample, "method": r.method, "cohort": meta[r.sample]["cohort"],
                 "overlap": 100 * n[2:].sum() / n[1:].sum(),
                 "low_k1": 100 * low[1] / n[1],
                 "low_k2": 100 * low[2:].sum() / n[2:].sum() if n[2:].sum() >= 1000 else np.nan,
                 "enrichment": np.log2((low[2:].sum() / n[2:].sum()) / (low[1] / n[1])) if n[2:].sum() >= 1000 else np.nan})
per_sample = pd.DataFrame(rows)
per_sample["delta"] = per_sample.low_k2 - per_sample.low_k1

ctrl = per_sample[per_sample.method == controls[0]].set_index("sample").enrichment
rng = np.random.default_rng(0)
stats = []
for m, d in per_sample.groupby("method"):
    e = d.set_index("sample").enrichment.dropna()
    boot = np.median(rng.choice(e.to_numpy(), (2000, len(e))), axis=1) if len(e) else [np.nan]
    paired = pd.concat([e, ctrl], axis=1, keys=["m", "c"]).dropna()
    stats.append({"method": m, "n": len(e), "overlap": d.overlap.median(), "enrichment": e.median(),
                  "ci_low": np.percentile(boot, 2.5), "ci_high": np.percentile(boot, 97.5),
                  "p_vs_0": wilcoxon(e).pvalue if len(e) >= 5 else np.nan,
                  "p_vs_control": wilcoxon(paired.m, paired.c).pvalue if m != controls[0] and len(paired) >= 5 else np.nan,
                  "low_k1": d.low_k1.median(), "low_k2": d.low_k2.median(),
                  "p_pair": wilcoxon(d.low_k2, d.low_k1, nan_policy="omit").pvalue if d.low_k2.notna().sum() >= 5 else np.nan})
stats = pd.DataFrame(stats).set_index("method").sort_values("overlap", ascending=False)
for col in ("p_vs_0", "p_vs_control", "p_pair"):
    ok = stats[col].notna()
    stats.loc[ok, col.replace("p_", "q_")] = false_discovery_control(stats.loc[ok, col])
stats.rename(index=method_label).round(3)

In [ ]:
sig = pd.cut(stats.q_vs_0, [0, 0.001, 0.01, 0.05, 1], labels=["***", "**", "*", ""]).astype(str).replace("nan", "")
y = np.arange(len(stats))
fig, axs = plt.subplots(1, 2, figsize=(110 * mm, (8 + 3.2 * len(stats)) * mm), sharey=True)
for ax, col in zip(axs, ("overlap", "enrichment")):
    vals = [per_sample.loc[per_sample.method == m, col].dropna().to_numpy() for m in stats.index]
    for yi, m, v in zip(y, stats.index, vals):
        ax.scatter(v, yi + rng.uniform(-0.15, 0.15, len(v)), s=2.5, color=method_color[m], alpha=0.75, lw=0, zorder=1)
    ax.vlines([np.median(v) if len(v) else np.nan for v in vals], y - 0.32, y + 0.32, color="k", lw=1.2, zorder=2)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)
axs[0].set(xscale="log", xlabel="Cell overlap (% of area)")
axs[0].xaxis.set_major_formatter(matplotlib.ticker.FormatStrFormatter("%g"))
axs[0].set_yticks(y, [method_label[m] for m in stats.index])
axs[0].set_ylim(len(stats) - 0.5, -0.5)
axs[1].axvline(0, color="0.35", lw=0.7, ls=(0, (3, 2)), zorder=0)
axs[1].set(xlabel="Low z integrity in overlaps (log2FC)")
for yi, m in zip(y, stats.index):
    axs[1].text(1.02, yi, sig[m], transform=axs[1].get_yaxis_transform(), va="center", fontsize=6)
fig.tight_layout(w_pad=0.3)
plt.show()

### Supplement: cohorts
Same as a and b, split by cohort. Low-VSI area in single cells differs between cohorts: about 20% in aging, 5% in ABCAtlas
and 3% in VizgenMouseBrain. The table gives medians per method and cohort.

In [ ]:
for col, xlabel in [("overlap", "Cell overlap (% of area)"),
                    ("enrichment", "Low z integrity in overlaps (log2FC)")]:
    fig, axs = plt.subplots(1, len(cohorts), figsize=(140 * mm, (8 + 3.2 * len(stats)) * mm), sharey=True, sharex=True)
    for ax, c in zip(axs, cohorts):
        d = per_sample[per_sample.cohort == c]
        vals = [d.loc[d.method == m, col].dropna().to_numpy() for m in stats.index]
        for yi, m, v in zip(y, stats.index, vals):
            ax.scatter(v, yi + rng.uniform(-0.15, 0.15, len(v)), s=2.5, color=method_color[m], alpha=0.75, lw=0, zorder=1)
        ax.vlines([np.median(v) if len(v) else np.nan for v in vals], y - 0.32, y + 0.32, color="k", lw=1.2, zorder=2)
        ax.set_title(f"{c} (n = {d['sample'].nunique()})")
        ax.spines["left"].set_visible(False)
        ax.tick_params(axis="y", length=0)
        if col == "enrichment":
            ax.axvline(0, color="0.35", lw=0.7, ls=(0, (3, 2)), zorder=0)
        else:
            ax.set_xscale("log")
            ax.xaxis.set_major_formatter(matplotlib.ticker.FormatStrFormatter("%g"))
    axs[0].set_yticks(y, [method_label[m] for m in stats.index])
    axs[0].set_ylim(len(stats) - 0.5, -0.5)
    axs[1].set_xlabel(xlabel)
    fig.tight_layout(w_pad=0.3)
    plt.show()


In [ ]:
per_sample.assign(method=per_sample.method.map(method_label)).pivot_table(
    index="method", columns="cohort", values=["low_k1", "low_k2", "enrichment"], aggfunc="median").loc[[method_label[m] for m in stats.index]].round(1)

### VSI shift in overlaps
Median VSI in overlap area minus single-cell area per sample, against the share of transcripts in overlapping cells.
Large dots are method medians. The dashed line is the shift expected if overlaps were vertical doublets.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(180 * mm, 62 * mm), sharex=True, sharey=True)
for ax, qc in zip(axs, ["pre", "post"]):
    dq = df[df.qc == qc]
    doublet = low_vsi - dq.vsi_k1.median()
    ax.axhline(0, color="0.6", lw=0.5)
    ax.axhline(doublet, color="0.2", ls=(0, (3, 2)), lw=0.6)
    ax.text(0.98, doublet, "expected for vertical doublets", transform=ax.get_yaxis_transform(),
            ha="right", va="bottom", fontsize=6, color="0.2")
    for m, d in dq.groupby("method"):
        ax.scatter(100 * d.k2_signal, d.dvsi, s=3, alpha=0.3, color=method_color[m], lw=0)
        ax.scatter(100 * d.k2_signal.median(), d.dvsi.median(), s=22, color=method_color[m], ec="white", lw=0.4,
                   zorder=3, label=method_label[m])
    ax.set(xscale="log", title=f"{qc.capitalize()}-QC", xlabel="Transcripts in overlapping cells (%)")
axs[0].set_ylabel("VSI difference, overlap minus single cell")
handles = dict(zip(*axs[1].get_legend_handles_labels()[::-1]))
axs[1].legend(handles.values(), handles.keys(), loc="center left", bbox_to_anchor=(1.02, 0.5))
fig.tight_layout()
plt.show()

## 2. Where overlaps sit
Crop with the most low-VSI signal. Black outlines pass QC, blue ones are removed by QC. Orange fill marks 2 cells, red 3 or more (before QC).

In [ ]:
%%time
npz = np.load(base / "samples" / show_sample / npz_file)
vsi, signal = npz["integrity"].astype(np.float32), npz["signal"].astype(np.float32)
tf = pd.read_csv(pathlib.Path(meta[show_sample]["path"], "images/micron_to_mosaic_pixel_transform.csv"),
                 sep=" ", header=None).values
low = ((signal >= min_signal) & (vsi < low_vsi)).astype(np.float32)
cy, cx = np.unravel_index(uniform_filter(low, crop).argmax(), low.shape)
y0, x0 = max(cy - crop // 2, 0), max(cx - crop // 2, 0)
win = np.s_[y0:y0 + crop, x0:x0 + crop]
methods = [m for m in order if m in methods_for(show_sample)]
ocmap = ListedColormap(["#f28e2b", "#c0392b"])

ncol = 4
nrow = -(-(len(methods) + 1) // ncol)
fig, axs = plt.subplots(nrow, ncol, figsize=(180 * mm, 180 * mm * nrow / ncol))
axs = axs.ravel()
axs[0].imshow(vsi[win], cmap="coolwarm_r", vmin=0, vmax=1, origin="lower", extent=(0, crop, 0, crop),
              alpha=(signal[win] / 2).clip(0, 1) ** 2)
axs[0].set_title("VSI")
for ax, m in zip(axs[1:], methods):
    foot, _, cells = load_cells(show_sample, m, tf, bbox=(x0, y0, x0 + crop, y0 + crop))
    passed = cells.keep.astype(bool)
    sub = foot.cx[x0:x0 + crop, y0:y0 + crop].translate(-x0, -y0)
    ok = passed[sub.index].to_numpy()
    k = count_map(sub.values, (crop, crop))
    ax.imshow(np.ma.masked_less(k.clip(0, 3), 2), cmap=ocmap, vmin=2, vmax=3, origin="lower",
              extent=(0, crop, 0, crop), interpolation="nearest")
    for part, col in [(sub[ok], "0.15"), (sub[~ok], "#3b75af")]:
        if len(part):
            part.boundary.plot(ax=ax, color=col, lw=0.25)
    after = 100 * (count_map(sub[ok].values, (crop, crop)) >= 2).mean()
    ax.set_title(f"{_constants.method_names.get(m, m)}\noverlap {100 * (k >= 2).mean():.1f}%, after QC {after:.1f}%", fontsize=6)
for ax in axs:
    ax.set(xlim=(0, crop), ylim=(0, crop), aspect="equal", xticks=[], yticks=[])
    ax.spines[:].set_visible(True)
    ax.spines[:].set_linewidth(0.3)
for ax in axs[len(methods) + 1:]:
    ax.axis("off")
fig.suptitle(f"{show_sample}, {crop} px crop")
fig.tight_layout()
plt.show()

## 3. Stacked cells are rare
Methods with z per polygon. Share of all and of low-VSI signal under both a cell confined to the top half and one confined
to the bottom half, and share of cells confined to either half.

In [ ]:
per_method[["zsplit_signal", "lowvsi_recall", "top_only", "bottom_only"]].dropna().mul(100).round(2)

## 4. Cells along z
Methods with z per polygon, pooled over samples; top row before QC, bottom row after.
Left: number of z planes per cell. Right: which section surface a cell reaches.

In [ ]:
%%time
import pyarrow.parquet as pq

zr_cache = base / "misc" / "tmp" / "z_overlap_zrange.pkl"
zr_raw = pd.read_pickle(zr_cache) if zr_cache.exists() else {}
for s in samples:
    for m in methods_for(s):
        if (s, m) in zr_raw:
            continue
        path = next((base / "samples" / s / "sdata_z3.zarr" / "shapes" / f"boundaries_{m}").glob("*.parquet"))
        cols = pq.read_schema(path).names
        zc = next((col for col in zcols if col in cols), None)
        if zc is None:
            continue
        d = pd.read_parquet(path, columns=[zc] + (["cell_id"] if "cell_id" in cols else []))
        ids = (d["cell_id"] if "cell_id" in d else d.index.to_series()).astype(str).to_numpy()
        zr = pd.to_numeric(d[zc]).groupby(ids).agg(["min", "max"])
        zr["keep"] = cell_table(s, m, ids).keep.reindex(zr.index).fillna(False).astype(bool)
        zr_raw[(s, m)] = zr.value_counts()
    pd.to_pickle(zr_raw, zr_cache)

zr = pd.concat({k: v for k, v in zr_raw.items() if k[0] in samples}, names=["sample", "method"]).groupby(["method", "min", "max", "keep"]).sum()
zr = pd.concat({"pre": zr.groupby(["method", "min", "max"]).sum(),
                "post": zr.xs(True, level="keep")}, names=["qc"]).rename("cells").reset_index()
zr["centre"] = (zr["min"] + zr["max"]) / 2
zr["planes"] = zr["max"] - zr["min"] + 1

In [ ]:
zorder = (zr[zr.qc == "post"].assign(full=lambda d: d.cells * (d.planes == 7)).groupby("method")[["full", "cells"]].sum()
          .eval("full / cells").sort_values(ascending=False).index)
reach_colors = {"Full depth": "#08306b", "Bottom only": "#e6550d", "Top only": "#31a354", "Neither": "#bdbdbd"}
y = np.arange(len(zorder))
fig, axs = plt.subplots(2, 2, figsize=(160 * mm, 110 * mm), sharex=True)
for row, qc in zip(axs, ("pre", "post")):
    dq = zr[zr.qc == qc]
    planes = dq.pivot_table(index="method", columns="planes", values="cells", aggfunc="sum", fill_value=0).loc[zorder]
    planes = 100 * planes.div(planes.sum(axis=1), axis=0)
    ax = row[0]
    bottom = np.zeros(len(planes))
    for col, colour in zip(planes.columns, plt.cm.YlGnBu(np.linspace(0.15, 0.95, planes.shape[1]))):
        ax.bar(y, planes[col], bottom=bottom, width=0.75, color=colour, lw=0, label=f"{col:g}")
        bottom += planes[col].to_numpy()
    ax.set(ylim=(0, 100), ylabel=f"Cells (%), {qc}-QC")
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis="x", length=0)

    lo, hi = zr["min"].min(), zr["max"].max()
    reach = np.select([(dq["min"] == lo) & (dq["max"] == hi), dq["min"] == lo, dq["max"] == hi],
                      ["Full depth", "Bottom only", "Top only"], "Neither")
    reach = (dq.assign(reach=reach).pivot_table(index="method", columns="reach", values="cells", aggfunc="sum", fill_value=0)
             .reindex(index=zorder, columns=list(reach_colors), fill_value=0))
    reach = 100 * reach.div(reach.sum(axis=1), axis=0)
    ax = row[1]
    bottom = np.zeros(len(reach))
    for col, colour in reach_colors.items():
        ax.bar(y, reach[col], bottom=bottom, width=0.75, color=colour, lw=0, label=col)
        bottom += reach[col].to_numpy()
    ax.set(ylim=(0, 100), ylabel=f"Cells (%), {qc}-QC")
    ax.spines["bottom"].set_visible(False)
    ax.tick_params(axis="x", length=0)
handles, labels = axs[0, 0].get_legend_handles_labels()
axs[0, 0].legend(handles[::-1], labels[::-1], title="z planes", loc="upper left", bbox_to_anchor=(1.01, 1),
                 handlelength=1, handletextpad=0.4)
handles, labels = axs[0, 1].get_legend_handles_labels()
axs[0, 1].legend(handles[::-1], labels[::-1], title="Reaches", loc="upper left", bbox_to_anchor=(1.01, 1),
                 handlelength=1, handletextpad=0.4)
for ax in axs[1]:
    ax.set_xticks(y, [_constants.method_names.get(m, m) for m in zorder], rotation=45, ha="right")
fig.tight_layout()
plt.show()

Lowest against highest z plane per cell. Cells in the top-right corner span the whole section, cells on the diagonal sit in a single plane.

In [ ]:
from matplotlib.colors import PowerNorm

mats = {(qc, m): zr[(zr.qc == qc) & (zr.method == m)]
                 .pivot_table(index="min", columns="max", values="cells", aggfunc="sum")
                 .reindex(index=range(7), columns=range(7))
        for qc in ("pre", "post") for m in zorder}
mats = {k: 100 * v / np.nansum(v.to_numpy()) for k, v in mats.items()}
norm = PowerNorm(0.5, vmin=0, vmax=max(np.nanmax(v.to_numpy()) for v in mats.values()))

fig, axs = plt.subplots(2, len(zorder), figsize=(180 * mm, 62 * mm), sharex=True, sharey=True, layout="constrained")
for row, qc in zip(axs, ("pre", "post")):
    for ax, m in zip(row, zorder):
        im = ax.imshow(mats[(qc, m)], cmap="Blues", norm=norm, origin="lower")
        ax.set(xticks=[0, 6], yticks=[0, 6])
        ax.spines[["top", "right"]].set_visible(True)
    row[0].set_ylabel(f"{qc.capitalize()}-QC")
for ax, m in zip(axs[0], zorder):
    ax.set_title("\n".join(_constants.method_names.get(m, m).rsplit(" ", 1)), fontsize=5)
fig.supxlabel("Highest z plane", fontsize=7)
fig.supylabel("Lowest z plane", fontsize=7)
fig.colorbar(im, ax=axs, shrink=0.8, label="Cells (%)")
plt.show()

## 5. Cells in 3D
Per-plane outlines after QC, coloured by cell type. Change `sample_3d`, `center_3d` (µm) or `size_3d` to move the region.

In [ ]:
%%time
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib
import shapely

sample_3d = show_sample
center_3d = (x0 + crop // 2, y0 + crop // 2)
size_3d, z_step, z_exag = 40, 1.5, 2   # region side (µm), µm per z plane, z stretch

tf3 = pd.read_csv(pathlib.Path(meta[sample_3d]["path"], "images/micron_to_mosaic_pixel_transform.csv"),
                  sep=" ", header=None).values
xmin, ymin = center_3d[0] - size_3d / 2, center_3d[1] - size_3d / 2
bx = shapely.box(xmin, ymin, xmin + size_3d, ymin + size_3d)
planes = {m: load_cells(sample_3d, m, tf3, planes=True, bbox=(xmin, ymin, xmin + size_3d, ymin + size_3d))
          for m in order if m in methods_for(sample_3d)}
planes = {m: p.query("keep") for m, p in planes.items() if p is not None}

ncol = min(4, len(planes))
nrow = -(-len(planes) // ncol)
fig = plt.figure(figsize=(180 * mm, 45 * mm * nrow))
for i, (m, pl) in enumerate(planes.items()):
    ax = fig.add_subplot(nrow, ncol, i + 1, projection="3d")
    z = pl.z * z_step if pl.z.max() <= 10 and (pl.z % 1 == 0).all() else pl.z
    verts, cols = [], []
    for g, zz, ct in zip(pl.geometry.intersection(bx), z, pl.cell_type):
        for poly in getattr(g, "geoms", [g]):
            if poly.geom_type == "Polygon" and not poly.is_empty:
                xy = np.asarray(poly.exterior.coords) - (xmin, ymin)
                verts.append(np.column_stack([xy, np.full(len(xy), zz * z_exag)]))
                cols.append(matplotlib.colors.to_rgb(_constants.cell_type_colors.get(ct, "#bdbdbd")))
    ax.add_collection3d(Poly3DCollection(verts, facecolors=[(*c, 0.35) for c in cols],
                                         edgecolors=[(*c, 0.9) for c in cols], linewidths=0.3))
    zmax = 7 * z_step * z_exag
    for xs, ys in [((0, size_3d), (0, 0)), ((0, 0), (0, size_3d))]:
        ax.plot(xs, ys, 0, color="0.5", lw=0.4)
    ax.plot((0, 0), (0, 0), (0, zmax), color="0.5", lw=0.4)
    ax.set(xlim=(0, size_3d), ylim=(0, size_3d), zlim=(0, zmax))
    ax.set_box_aspect((1, 1, zmax / size_3d), zoom=1.1)
    ax.set_proj_type("ortho")
    ax.view_init(elev=20, azim=-60)
    ax.set_axis_off()
    ax.set_title(f"{_constants.method_names.get(m, m)}\n{pl.cell.nunique()} cells", fontsize=6, pad=0)
fig.suptitle(f"{sample_3d}, {size_3d} x {size_3d} µm, z stretched {z_exag}-fold; grey axes {size_3d} µm and {7 * z_step:.1f} µm", fontsize=7)
types = [t for t in _constants.cell_type_colors if any((p.cell_type == t).any() for p in planes.values())]
fig.legend(handles=[matplotlib.patches.Patch(color=_constants.cell_type_colors[t], label=t) for t in types],
           loc="lower center", ncol=min(6, len(types)), bbox_to_anchor=(0.5, 0), handlelength=1)
fig.subplots_adjust(wspace=0.02, hspace=0.05, top=0.82, bottom=0.08)
plt.show()

The same region with all z planes overlaid.

In [ ]:
fig, axs = plt.subplots(nrow, ncol, figsize=(180 * mm, 45 * mm * nrow))
axs = axs.ravel()
for ax, (m, pl) in zip(axs, planes.items()):
    pl.geometry.intersection(bx).translate(-xmin, -ymin).boundary.plot(ax=ax, color="k", lw=0.3)
    ax.set(xlim=(0, size_3d), ylim=(0, size_3d), aspect="equal", xticks=[], yticks=[])
    ax.spines[:].set_visible(True)
    ax.spines[:].set_linewidth(0.3)
    ax.set_title(_constants.method_names.get(m, m), fontsize=6)
for ax in axs[len(planes):]:
    ax.axis("off")
fig.suptitle(f"{sample_3d}, {size_3d} x {size_3d} µm, all z planes", fontsize=7)
fig.tight_layout()
plt.show()